# ZAC Reverse Compiler — Notebook Test

This notebook exercises the reverse compiler end-to-end. It reconstructs a
**physical-qubit** circuit from a ZAC hardware schedule (ZAIR) and verifies the
core semantics:

- atom **movement** updates position only — it emits no gate;
- exchanging two atoms' spatial positions does **not** emit `SWAP`;
- only an explicit quantum `swap` emits a `SWAP` gate;
- export to **Qiskit** (arbitrary rotations) and **Stim** (Clifford-only).

## Setup
Make the standalone `reverse_compiler` package importable, and add the sibling
`ZAC-main/` checkout to the path for the round-trip test.

In [1]:
import os
import sys

# Notebook lives in the package root: .../Compiler/zac_reverse_compiler/
HERE = os.getcwd()
COMPILER_ROOT = os.path.dirname(HERE)
ZAC_MAIN = os.path.join(COMPILER_ROOT, "ZAC-main")
for p in (HERE, ZAC_MAIN):
    if p not in sys.path:
        sys.path.insert(0, p)

print("package root :", HERE)
print("ZAC-main     :", ZAC_MAIN, "(exists:", os.path.isdir(ZAC_MAIN), ")")

package root : /Users/bryantduan/Desktop/Compiler/Reverse_Compiler
ZAC-main     : /Users/bryantduan/Desktop/Compiler/ZAC-main (exists: True )


In [2]:
import math

from reverse_compiler import (
    CircuitIR,
    ReverseCompiler,
    StimExportError,
    encode_circuit,
    reverse_compile,
    to_qiskit,
    to_stim,
)
from reverse_compiler.reverse_compiler import ReverseCompileError

# Positions are (array/SLM id, row, col). array 0 = storage, array 1 = entangling.
STORAGE, ENTANGLE = 0, 1


def make_init(n_atoms, array=STORAGE):
    """A ZAIR 'init' placing n_atoms in a row."""
    return {
        "type": "init",
        "id": 0,
        "begin_time": 0,
        "end_time": 0,
        "init_locs": [[i, array, 0, i] for i in range(n_atoms)],
    }


def zair(*instructions, name="demo", **extra):
    base = {"name": name, "architecture_spec_path": None, "instructions": list(instructions)}
    base.update(extra)
    return base


def show(ir):
    return [str(op) for op in ir.operations]


print("imports OK")

imports OK


## 1. Local single-qubit gates

In [3]:
code = zair(
    make_init(2),
    {"type": "1qGate", "unitary": "u3", "gates": [{"name": "h", "q": 0}, {"name": "x", "q": 1}]},
)
ir = reverse_compile(code)
print(show(ir))
assert [(op.name, op.qubits) for op in ir] == [("H", (0,)), ("X", (1,))]

['H q0', 'X q1']


## 2. Global gate — only the affected/active atoms
A global pulse restricted to the entangling zone hits only the atoms in it.

In [4]:
code = zair(
    {"type": "init", "init_locs": [[0, STORAGE, 0, 0], [1, STORAGE, 0, 1], [2, ENTANGLE, 0, 0]]},
    {"type": "global_1qGate", "name": "z", "zone": ENTANGLE},
)
ir = reverse_compile(code)
print(show(ir))
assert [op.qubits for op in ir] == [(2,)]

['Z q2']


## 3 & 4. CZ without and with movement
Movement into/out of the entangling zone emits **no** gate; only the `CZ` survives,
and the qubit↔atom association is unchanged.

In [5]:
# CZ without movement
code = zair(make_init(2), {"type": "rydberg", "zone_id": 0, "gates": [{"q0": 0, "q1": 1}]})
assert [(op.name, op.qubits) for op in reverse_compile(code)] == [("CZ", (0, 1))]

# CZ with movement (in, gate, out)
code = zair(
    make_init(2),
    {"type": "rearrangeJob", "aod_qubits": [0, 1],
     "begin_locs": [[0, STORAGE, 0, 0], [1, STORAGE, 0, 1]],
     "end_locs": [[0, ENTANGLE, 0, 0], [1, ENTANGLE, 0, 1]]},
    {"type": "rydberg", "zone_id": 0, "gates": [{"q0": 0, "q1": 1}]},
    {"type": "rearrangeJob", "aod_qubits": [0, 1],
     "begin_locs": [[0, ENTANGLE, 0, 0], [1, ENTANGLE, 0, 1]],
     "end_locs": [[0, STORAGE, 0, 0], [1, STORAGE, 0, 1]]},
)
rc = ReverseCompiler()
ir = rc.compile(code)
print("ops        :", show(ir))
print("qubit->atom:", rc.state.qubit_to_atom)
print("atom 0 pos :", rc.state.atom_to_position[0])
assert [(op.name, op.qubits) for op in ir] == [("CZ", (0, 1))]
assert rc.state.qubit_to_atom == {0: 0, 1: 1}

ops        : ['CZ q0 q1']
qubit->atom: {0: 0, 1: 1}
atom 0 pos : (0, 0, 0)


## 5. Coordinate-targeted gate
The gate names a hardware coordinate; the compiler resolves position → atom → qubit.

In [6]:
code = zair(
    make_init(3),
    {"type": "1qGate", "gates": [{"name": "h", "position": [STORAGE, 0, 2]}]},
)
ir = reverse_compile(code)
print(show(ir))
assert [(op.name, op.qubits) for op in ir] == [("H", (2,))]

['H q2']


## 6. Parallel CZ gates (same layer)

In [7]:
code = zair(
    make_init(4),
    {"type": "rydberg", "zone_id": 0, "gates": [{"q0": 0, "q1": 1}, {"q0": 2, "q1": 3}]},
)
ir = reverse_compile(code)
print(show(ir))
assert ir.operations[0].layer == ir.operations[1].layer
assert to_stim(ir).num_ticks == 0  # both gates in one moment

['CZ q0 q1', 'CZ q2 q3']


## 7. Atoms exchanging positions — no SWAP
A spatial exchange updates `position_to_atom` but produces zero quantum gates.

In [8]:
code = zair(
    make_init(2),
    {"type": "rearrangeJob", "aod_qubits": [0, 1],
     "begin_locs": [[0, STORAGE, 0, 0], [1, STORAGE, 0, 1]],
     "end_locs": [[0, STORAGE, 0, 1], [1, STORAGE, 0, 0]]},
)
rc = ReverseCompiler()
ir = rc.compile(code)
print("num ops          :", len(ir))
print("atom 0 position  :", rc.state.atom_to_position[0])
print("position->atom   :", rc.state.position_to_atom)
assert len(ir) == 0
assert rc.state.atom_to_position[0] == (STORAGE, 0, 1)
assert rc.state.position_to_atom[(STORAGE, 0, 1)] == 0

num ops          : 0
atom 0 position  : (0, 0, 1)
position->atom   : {(0, 0, 1): 0, (0, 0, 0): 1}


## 8. Explicit quantum SWAP

In [9]:
code = zair(make_init(2), {"type": "swap", "qubits": [0, 1]})
ir = reverse_compile(code)
print(show(ir))
assert [(op.name, op.qubits) for op in ir] == [("SWAP", (0, 1))]

['SWAP q0 q1']


## 9. Measurement and reset

In [10]:
code = zair(
    make_init(2),
    {"type": "reset", "qubits": [0]},
    {"type": "measure", "qubits": [0, 1], "classical_bits": [0, 1]},
)
ir = reverse_compile(code)
print(show(ir))
assert [op.name for op in ir] == ["RESET", "MEASURE", "MEASURE"]
qc = to_qiskit(ir)
print(qc)

['RESET q0', 'MEASURE q0 -> c0', 'MEASURE q1 -> c1']
          ┌─┐
q_0: ─|0>─┤M├
      ┌─┐ └╥┘
q_1: ─┤M├──╫─
      └╥┘  ║ 
c: 2/══╩═══╩═
       1   0 


## 10. Arbitrary rotations in Qiskit

In [11]:
code = zair(
    make_init(2),
    {"type": "1qGate", "gates": [
        {"name": "rz", "q": 0, "params": [0.37]},
        {"name": "u3", "q": 1, "params": [0.1, 0.2, 0.3]},
    ]},
)
ir = reverse_compile(code)
print(show(ir))
qc = to_qiskit(ir)
print(qc)
assert [op.name for op in ir] == ["RZ", "U"]

['RZ(0.37) q0', 'U(0.1, 0.2, 0.3) q1']
        ┌──────────┐   
q_0: ───┤ Rz(0.37) ├───
     ┌──┴──────────┴──┐
q_1: ┤ U(0.1,0.2,0.3) ├
     └────────────────┘


## 11. Stim rejects non-Clifford rotations; simplifies Clifford ones

In [12]:
# Non-Clifford RZ(0.37) -> rejected for Stim (but fine for Qiskit)
ir = reverse_compile(zair(make_init(1), {"type": "1qGate", "gates": [{"name": "rz", "q": 0, "params": [0.37]}]}))
to_qiskit(ir)
try:
    to_stim(ir)
    raise AssertionError("expected StimExportError")
except StimExportError as exc:
    print("rejected as expected:", exc)

# Exact-Clifford rotations are simplified
code = zair(make_init(4), {"type": "1qGate", "gates": [
    {"name": "rx", "q": 0, "params": [math.pi]},
    {"name": "ry", "q": 1, "params": [math.pi]},
    {"name": "rz", "q": 2, "params": [math.pi / 2]},
    {"name": "rz", "q": 3, "params": [-math.pi / 2]},
]})
stim_circuit = to_stim(reverse_compile(code))
print(stim_circuit)
text = str(stim_circuit)
assert "X 0" in text and "Y 1" in text and "S 2" in text and "S_DAG 3" in text

rejected as expected: non-Clifford rotation RZ(0.37) q0 cannot be exported to Stim
X 0
Y 1
S 2
S_DAG 3


## 12. Lost / inactive atoms
A lost atom is excluded from global pulses.

In [13]:
code = zair(
    make_init(3),
    {"type": "loss", "atoms": [2]},
    {"type": "global_1qGate", "name": "h"},
)
rc = ReverseCompiler()
ir = rc.compile(code)
print(show(ir))
print("active atoms:", rc.state.active_atoms)
assert [op.qubits for op in ir] == [(0,), (1,)]
assert 2 not in rc.state.active_atoms

['H q0', 'H q1']
active atoms: {0, 1}


## Round trip through the real ZAC compiler
`physical circuit → ZAC → schedule → reverse compiler → reconstructed circuit`,
compared after removing hardware-only operations.

In [6]:
import json
import tempfile

arch_spec = os.path.join(ZAC_MAIN, "hardware_spec", "full_architecture.json")

from zac.ds.architecture import Architecture
from zac.zac import ZAC

qasm = (
    'OPENQASM 2.0;\n'
    'include "qelib1.inc";\n'
    'qreg q[4];\n'
    'h q[0];\nh q[1];\nh q[2];\nh q[3];\n'
    'cz q[0],q[1];\n'
    'cz q[2],q[3];\n'
    'cz q[1],q[2];\n'
)
with tempfile.TemporaryDirectory() as d:
    qpath = os.path.join(d, "rt.qasm")
    with open(qpath, "w") as f:
        f.write(qasm)
    with open(arch_spec) as f:
        spec = json.load(f)
    arch = Architecture(spec)
    arch.preprocessing()
    compiler = ZAC()
    compiler.parse_setting({"name": "rt", "reuse": False, "resyn": False,
                            "trivial_placement": True, "use_window": False, "use_verifier": False})
    compiler.set_architecture_spec_path(arch_spec)
    compiler.set_architecture(arch)
    compiler.set_program(qpath)
    code_dict = compiler.solve(save_file=False)

ir = reverse_compile(code_dict)


def signature(ir):
    counts = {}
    for op in ir.without_timing():
        if op.name == "CZ":
            key = ("CZ", frozenset(op.qubits))
        elif op.name == "H":
            key = ("H", op.qubits[0])
        else:
            key = (op.name, op.qubits)
        counts[key] = counts.get(key, 0) + 1
    return counts


original = {
    ("CZ", frozenset({0, 1})): 1,
    ("CZ", frozenset({1, 2})): 1,
    ("CZ", frozenset({2, 3})): 1,
    ("H", 0): 1, ("H", 1): 1, ("H", 2): 1, ("H", 3): 1,
}
reconstructed = signature(ir)
print("reconstructed:", reconstructed)
assert reconstructed == original
print("\nround-trip OK (gate multiset)")

# --- Stronger check: full unitary equivalence (global-phase tolerant) -------
# The gate multiset ignores ordering. Comparing the *unitary* of the original
# circuit against the reconstructed one verifies the actual semantics, tolerating
# any commuting-gate reordering or routing ZAC may have applied.
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

qc_orig = QuantumCircuit(4)
for q in range(4):
    qc_orig.h(q)
qc_orig.cz(0, 1)
qc_orig.cz(2, 3)
qc_orig.cz(1, 2)

qc_recon = to_qiskit(ir)
assert Operator(qc_orig).equiv(Operator(qc_recon)), "unitaries differ"
print("round-trip OK (unitary equivalence, global-phase tolerant)")
print(qc_recon)


[INFO] ZAC: Parse Circuit
[INFO]           number of qubits: 4
[INFO]           number of two-qubit gates: 3
[INFO]           number of single-qubit gates: 4
[INFO] ZAC: A compiler for neutral atom-based compute-store architecture
[INFO] ZAC: Setting
[INFO]           Result directory: ./result/
[INFO]           Scheduling strategy: asap
[INFO]           Placement strategy: trivial placement
[INFO]           Intermediate placement strategy: minimal weighted matching
[INFO]                                         : no reuse
[INFO]           Routing strategy: maximalis_sort without window
[INFO]           Verifier: disable
[INFO] ZAC: Run scheduling
[INFO]               Time for scheduling: 1.0967254638671875e-05s
[INFO]               Time for initial placement: 4.6253204345703125e-05s
[INFO] ZAC: Minimum-weight-full-matching-based intermediate placement: Start
[INFO] ZAC: Minimum-weight-full-matching-based intermediate placement: Finish
[INFO]               Time for intermediate placemen

## Larger round trip through the real ZAC compiler
A wider, deeper circuit — **12 qubits, multi-layer GHZ-style entanglement** —
compiled by the real ZAC (`resyn=False` so gate names survive), then reverse
compiled. The reconstruction is compared to the original as a gate multiset
(ZAC is free to reorder/route, so we match counts of `H`/`CZ` rather than order).

In [7]:
import json
import tempfile
from collections import Counter

from zac.ds.architecture import Architecture
from zac.zac import ZAC

N = 12  # qubits

# Build a wide, deep circuit programmatically: H layer, several CZ layers, H layer.
h_front = [("h", (q,)) for q in range(N)]
cz_layer1 = [("cz", (i, i + 1)) for i in range(0, N - 1, 2)]      # (0,1)(2,3)...(10,11)
cz_layer2 = [("cz", (i, i + 1)) for i in range(1, N - 1, 2)]      # (1,2)(3,4)...(9,10)
cz_layer3 = [("cz", (0, 11)), ("cz", (2, 9)), ("cz", (4, 7))]     # long-range
h_back = [("h", (q,)) for q in range(N)]
gates = h_front + cz_layer1 + cz_layer2 + cz_layer3 + h_back

# Emit QASM 2.0 for ZAC.
lines = ['OPENQASM 2.0;', 'include "qelib1.inc";', f'qreg q[{N}];']
for name, qs in gates:
    if len(qs) == 1:
        lines.append(f"{name} q[{qs[0]}];")
    else:
        lines.append(f"{name} q[{qs[0]}],q[{qs[1]}];")
qasm = "\n".join(lines) + "\n"

# Expected gate multiset.
original = Counter()
for name, qs in gates:
    if name == "cz":
        original[("CZ", frozenset(qs))] += 1
    elif name == "h":
        original[("H", qs[0])] += 1
print("original gate count:", sum(original.values()),
      "(H:", len(h_front) + len(h_back), "CZ:", len(cz_layer1 + cz_layer2 + cz_layer3), ")")

arch_spec = os.path.join(ZAC_MAIN, "hardware_spec", "full_architecture.json")
with tempfile.TemporaryDirectory() as d:
    qpath = os.path.join(d, "rt_big.qasm")
    with open(qpath, "w") as f:
        f.write(qasm)
    with open(arch_spec) as f:
        spec = json.load(f)
    arch = Architecture(spec)
    arch.preprocessing()
    compiler = ZAC()
    compiler.parse_setting({"name": "rt_big", "reuse": False, "resyn": False,
                            "trivial_placement": True, "use_window": False, "use_verifier": False})
    compiler.set_architecture_spec_path(arch_spec)
    compiler.set_architecture(arch)
    compiler.set_program(qpath)
    code_dict = compiler.solve(save_file=False)

ir = reverse_compile(code_dict)


def signature(ir):
    counts = Counter()
    for op in ir.without_timing():
        if op.name == "CZ":
            counts[("CZ", frozenset(op.qubits))] += 1
        elif op.name == "H":
            counts[("H", op.qubits[0])] += 1
        else:
            counts[(op.name, op.qubits)] += 1
    return counts


reconstructed = signature(ir)
print("reconstructed gate count:", sum(reconstructed.values()))
assert reconstructed == original
print("\nlarger ZAC round-trip OK (gate multiset)")

# --- Stronger check: Stim tableau equivalence ------------------------------
# This circuit is pure Clifford (H + CZ), so a 4096x4096 unitary is wasteful.
# Instead we compare stabilizer *tableaux*, which is exact and scales far past
# what dense unitaries allow, while still catching ordering/routing errors.
import stim


def gates_to_stim(gate_list, n):
    c = stim.Circuit()
    for nm, qs in gate_list:
        if nm == "h":
            c.append("H", [qs[0]])
        elif nm == "cz":
            c.append("CZ", [qs[0], qs[1]])
        else:
            raise ValueError(f"unexpected non-Clifford gate {nm}")
    # ensure every qubit is referenced so both tableaux have the same size
    for q in range(n):
        c.append("I", [q])
    return c


orig_tableau = gates_to_stim(gates, N).to_tableau()
recon_tableau = to_stim(ir).to_tableau()
assert orig_tableau == recon_tableau, "stabilizer tableaux differ"
print("larger ZAC round-trip OK (Stim tableau equivalence, 12 qubits)")


original gate count: 38 (H: 24 CZ: 14 )
[INFO] ZAC: Parse Circuit
[INFO]           number of qubits: 12
[INFO]           number of two-qubit gates: 14
[INFO]           number of single-qubit gates: 24
[INFO] ZAC: A compiler for neutral atom-based compute-store architecture
[INFO] ZAC: Setting
[INFO]           Result directory: ./result/
[INFO]           Scheduling strategy: asap
[INFO]           Placement strategy: trivial placement
[INFO]           Intermediate placement strategy: minimal weighted matching
[INFO]                                         : no reuse
[INFO]           Routing strategy: maximalis_sort without window
[INFO]           Verifier: disable
[INFO] ZAC: Run scheduling
[INFO]               Time for scheduling: 9.059906005859375e-06s
[INFO]               Time for initial placement: 3.4809112548828125e-05s
[INFO] ZAC: Minimum-weight-full-matching-based intermediate placement: Start
[INFO] ZAC: Minimum-weight-full-matching-based intermediate placement: Finish
[INFO]   

## Round trip (self-contained, no ZAC dependency)
`physical circuit → reference encoder → hardware schedule → reverse compiler → reconstructed circuit`.

This uses the package's own `encode_circuit` (which borrows ZAC's *idea* — atoms moved
between storage and entangling zones — but no ZAC code). Because our hardware format
records single-qubit **rotation angles**, even arbitrary rotations reconstruct exactly.

In [15]:
# Original physical circuit, including arbitrary-angle rotations.
ops = [
    ("h", (0,)),
    ("rz", (0,), (0.37,)),
    ("rx", (1,), (1.2,)),
    ("cz", (0, 1)),          # realised in the schedule via movement
    ("ry", (2,), (-0.8,)),
    ("cz", (1, 2)),
]

# physical circuit -> hardware schedule (reference encoder, no ZAC) -> reverse compiler
schedule = encode_circuit(ops, n_qubits=3)
print("schedule instruction types:", [i["type"] for i in schedule["instructions"]])

ir = reverse_compile(schedule)
print("reconstructed:", show(ir))

CANON = {"h": "H", "rx": "RX", "ry": "RY", "rz": "RZ", "cz": "CZ"}


def expected(ops):
    out = []
    for name, qubits, *rest in ops:
        params = tuple(rest[0]) if rest else ()
        out.append((CANON[name], tuple(qubits), tuple(round(p, 9) for p in params)))
    return out


def reconstructed(ir):
    return [(op.name, tuple(op.qubits), tuple(round(p, 9) for p in op.params))
            for op in ir.without_timing()]


assert reconstructed(ir) == expected(ops)
print("\nself-contained round-trip OK (rotation angles preserved)")
print(to_qiskit(ir))

schedule instruction types: ['init', '1qGate', '1qGate', '1qGate', 'rearrangeJob', 'rydberg', 'rearrangeJob', '1qGate', 'rearrangeJob', 'rydberg', 'rearrangeJob']
reconstructed: ['H q0', 'RZ(0.37) q0', 'RX(1.2) q1', 'CZ q0 q1', 'RY(-0.8) q2', 'CZ q1 q2']

self-contained round-trip OK (rotation angles preserved)
        ┌───┐    ┌──────────┐      
q_0: ───┤ H ├────┤ Rz(0.37) ├─■────
     ┌──┴───┴──┐ └──────────┘ │    
q_1: ┤ Rx(1.2) ├──────────────■──■─
     ├─────────┴┐                │ 
q_2: ┤ Ry(-0.8) ├────────────────■─
     └──────────┘                  


## Complex round trip (self-contained, no ZAC dependency)
A larger stress test: **8 qubits, 20 gates** spanning the full supported gate set
(`h, x, y, z, rx, ry, rz, cz, cx, swap`), interleaved with the storage⇄entangling
movement that realises every two-qubit gate. The reconstruction must match the
original circuit exactly — order, qubits, and rotation angles included.

In [3]:
# A larger physical circuit: 8 qubits, 20 gates, full supported gate set.
big_ops = [
    ("h", (0,)),
    ("h", (1,)),
    ("h", (2,)),
    ("h", (3,)),
    ("x", (4,)),
    ("y", (5,)),
    ("z", (6,)),
    ("rx", (7,), (0.5,)),
    ("cz", (0, 1)),
    ("cx", (2, 3)),
    ("ry", (4,), (1.1,)),
    ("rz", (5,), (-0.7,)),
    ("cz", (4, 5)),
    ("swap", (6, 7)),       # explicit quantum swap (not a spatial move)
    ("cx", (1, 2)),
    ("h", (7,)),
    ("rz", (0,), (0.25,)),
    ("cz", (3, 4)),
    ("ry", (6,), (2.0,)),
    ("cx", (0, 7)),
]

schedule = encode_circuit(big_ops, n_qubits=8)
inst_types = [i["type"] for i in schedule["instructions"]]
print("schedule length      :", len(inst_types))
print("rearrangeJob count   :", inst_types.count("rearrangeJob"))
print("rydberg (CZ) count   :", inst_types.count("rydberg"))

ir = reverse_compile(schedule)
print("reconstructed gates  :", len(ir))

BIG_CANON = {
    "h": "H", "x": "X", "y": "Y", "z": "Z",
    "rx": "RX", "ry": "RY", "rz": "RZ",
    "cz": "CZ", "cx": "CX", "swap": "SWAP",
}


def big_expected(ops):
    out = []
    for name, qubits, *rest in ops:
        params = tuple(rest[0]) if rest else ()
        out.append((BIG_CANON[name], tuple(qubits), tuple(round(p, 9) for p in params)))
    return out


def big_reconstructed(ir):
    return [(op.name, tuple(op.qubits), tuple(round(p, 9) for p in op.params))
            for op in ir.without_timing()]


assert big_reconstructed(ir) == big_expected(big_ops)
print("\ncomplex round-trip OK (8 qubits, 20 gates, angles preserved)")

# Exporters must also succeed on the reconstructed circuit.
qc = to_qiskit(ir)
assert qc.num_qubits == 8
print(qc)


schedule length      : 33
rearrangeJob count   : 12
rydberg (CZ) count   : 6
reconstructed gates  : 20

complex round-trip OK (8 qubits, 20 gates, angles preserved)
        ┌───┐               ┌──────────┐        
q_0: ───┤ H ├────────■──────┤ Rz(0.25) ├─────■──
        ├───┤        │      └──────────┘     │  
q_1: ───┤ H ├────────■───────────■───────────┼──
        ├───┤                  ┌─┴─┐         │  
q_2: ───┤ H ├────────■─────────┤ X ├─────────┼──
        ├───┤      ┌─┴─┐       └───┘         │  
q_3: ───┤ H ├──────┤ X ├─────────────────■───┼──
        ├───┤   ┌──┴───┴──┐              │   │  
q_4: ───┤ X ├───┤ Ry(1.1) ├──────■───────■───┼──
        ├───┤   ├─────────┴┐     │           │  
q_5: ───┤ Y ├───┤ Rz(-0.7) ├─────■───────────┼──
        ├───┤   └──────────┘ ┌───────┐       │  
q_6: ───┤ Z ├────────X───────┤ Ry(2) ├───────┼──
     ┌──┴───┴──┐     │       └─┬───┬─┘     ┌─┴─┐
q_7: ┤ Rx(0.5) ├─────X─────────┤ H ├───────┤ X ├
     └─────────┘               └───┘       └───┘


## All checks passed ✅

In [16]:
print("All notebook tests passed.")

All notebook tests passed.
